# Del clima a la generación real: plantas solares de Colombia

Pregunta: ¿cuánto de la variación diaria de la generación de una planta solar colombiana se explica con el clima de ese día?

El cuaderno baja la generación real desde XM, la empareja con las plantas del dataset de Mantilla-Guerra et al. (2026), trae el clima
de cada sitio y compara varios modelos. Las referencias completas se encuentran al final del documento.

Por qué no usamos `solar_aptitude`: el índice del dataset sale solo de pendiente, orientación, sombra y curvatura (Ec. 1 del paper), así que
el clima no puede predecirlo. El factor de capacidad real sí depende del clima por física, y eso se puede medir.

## Pipeline

De la generación real de XM y el clima de Open-Meteo a la base diaria de Colombia.

![Pipeline](fig_pipeline_clima.png)

## 1. Fuentes y corte de tiempo

| Fuente | Qué aporta |
|---|---|---|
| XM, API pública `servapibi.xm.com.co` (XM, 2026) | Generación horaria por planta (`Gene`), capacidad efectiva neta diaria (`CapEfecNeta`) |
| Dataset del paper, con la tabla de GEM de febrero de 2026 (Mantilla-Guerra et al., 2026; GEM, 2026) | Coordenadas, nombre y capacidad de cada planta |
| Open-Meteo, API histórica (Zippenfenig, 2023) | Radiación, nubosidad, temperatura, humedad, viento y radiación en plano inclinado, por hora |

Corte de tiempo: el paper usa el release de GEM de febrero de 2026, el potencial de Global Solar Atlas del período 2018-2024 y carreteras de
OSM del 14 de marzo de 2026. No dice qué años promedia ERA5. La ventana de este cuaderno termina el 28 de febrero de 2026, el mes del release
de GEM, y empieza el 1 de enero de 2024. El inicio es decisión nuestra.

El clima no es una medición en el sitio. Open-Meteo entrega el valor de la celda de un modelo. Comparamos dos productos con la misma consulta:
`era5`, de 0.25° y documentado en la literatura, y `best_match`, el valor por defecto, que en las pruebas cayó en una celda más fina (ver sección 6).

In [ ]:
import json, re, time, unicodedata, difflib, warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr, wilcoxon
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")

SEED = 42
RNG = np.random.default_rng(SEED)
START, END = date(2024, 1, 1), date(2026, 2, 28)
CAP_REF_DATE = "2026-02-15"      # fecha para decidir qué plantas pasan de 1 MW
MIN_MW = 1.0
RAMP_DAYS = 60                   # días de puesta en marcha que se excluyen en el análisis principal
MIN_DIAS = 100                   # una planta con menos días útiles que esto no entra a la evaluación
CLUSTER_KM = 100                 # plantas a menos de esta distancia comparten clima y se tratan como un grupo
XM_URL = "https://servapibi.xm.com.co/"
OM_URL = "https://archive-api.open-meteo.com/v1/archive"
DATA = Path("datos_xm")
DATA.mkdir(exist_ok=True)
GEM_CSV = Path("..") / "Dataset_Mundial_Final(2).csv"

## 2. Generación real desde XM

La API acepta rangos de hasta 30 días en datos horarios y diarios. Cada descarga se guarda en `datos_xm/` y, si el archivo ya existe, no se
vuelve a pedir.

Primero se listan los recursos de XM con fuente solar. Nos quedamos con los que tenían al menos 1 MW de capacidad efectiva neta en la fecha
de referencia. Los autogeneradores pequeños (`AGPE`) quedan fuera por tamaño.

In [ ]:
def xm_post(endpoint, body, tries=3):
    for _ in range(tries):
        try:
            r = requests.post(XM_URL + endpoint, json=body, timeout=180)
            if r.ok:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError(f"XM no respondió para {endpoint} {body.get('MetricId')}")

path_rec = DATA / "recursos_solares.csv"
if path_rec.exists():
    solares = pd.read_csv(path_rec)
else:
    lista = xm_post("lists", {"MetricId": "ListadoRecursos"})
    todos = pd.DataFrame([it["ListEntities"][0]["Values"] for it in lista["Items"]])
    es_solar = todos.EnerSource.str.contains("RAD|SOL", na=False) | todos.Type.str.contains("SOLAR", case=False, na=False)
    solares = todos[es_solar].copy()
    solares.to_csv(path_rec, index=False)

print(f"Recursos solares registrados en XM: {len(solares)}")
print("Autogeneradores o generación distribuida (AGPE, GD):", int(solares.Name.str.match(r"^(AGPE|GD) ").sum()))

In [ ]:
def capacidad(ini, fin, codigos):
    j = xm_post("daily", {"MetricId": "CapEfecNeta", "StartDate": str(ini), "EndDate": str(fin),
                          "Entity": "Recurso", "Filter": codigos})
    return [{"date": it["Date"], "code": e["Code"], "cap_MW": float(e["Value"]) / 1000}   # la API entrega kW
            for it in j["Items"] for e in it["DailyEntities"] if e.get("Value") not in (None, "")]

ref = pd.DataFrame(capacidad(CAP_REF_DATE, CAP_REF_DATE, solares.Code.tolist()))
codigos = sorted(ref[ref.cap_MW >= MIN_MW].code)
print(f"Plantas con al menos {MIN_MW} MW al {CAP_REF_DATE}: {len(codigos)}")

In [ ]:
def descarga_por_tramos(ruta, pedir):
    if ruta.exists():
        return pd.read_csv(ruta)
    filas, ini = [], START
    while ini <= END:
        fin = min(ini + timedelta(days=29), END)
        filas.extend(pedir(ini, fin))
        ini = fin + timedelta(days=1)
    df = pd.DataFrame(filas)
    df.to_csv(ruta, index=False)
    return df

def pedir_generacion(ini, fin):
    j = xm_post("hourly", {"MetricId": "Gene", "StartDate": str(ini), "EndDate": str(fin),
                           "Entity": "Recurso", "Filter": codigos})
    return [{"date": it["Date"], **ent["Values"]} for it in j["Items"] for ent in it["HourlyEntities"]]

gen = descarga_por_tramos(DATA / "generacion_horaria.csv", pedir_generacion)
cap_dia = descarga_por_tramos(DATA / "capacidad_diaria.csv", lambda a, b: capacidad(a, b, codigos))

horas = [c for c in gen.columns if c.startswith("Hour")]
gen[horas] = gen[horas].apply(pd.to_numeric, errors="coerce")
gen["date"] = pd.to_datetime(gen["date"])
gen["kwh"] = gen[horas].sum(axis=1)
cap_dia["date"] = pd.to_datetime(cap_dia["date"])
print("Generación:", gen.shape, "| capacidad diaria:", cap_dia.shape, "| plantas:", gen.code.nunique())

## 3. Factor de capacidad diario

El factor de capacidad (FC) es la energía del día dividida entre la que habría entregado la planta a potencia nominal las 24 horas:

`FC = kWh del día / (capacidad efectiva neta en kW × 24)`

La capacidad es la de ese mismo día, porque varias plantas crecen por etapas. Es importante usar la capacidad del mismo día para evitar saltos artificiales cuando la planta entra en nuevas etapas de operación. Un FC mayor a 1.05 se descarta como dato roto.

Puesta en marcha: en una planta cuyo primer día con generación cae dentro de la ventana, los primeros `RAMP_DAYS` días se excluyen del análisis
principal. El criterio no mira ni la radiación ni el FC. Una planta que ya operaba el 1 de enero de 2024 no tiene su arranque observado y no pierde días.

In [ ]:
diario = gen[["code", "date", "kwh"]].merge(cap_dia, on=["code", "date"], how="inner")
diario["fc"] = diario.kwh / 1000 / (diario.cap_MW * 24)
diario = diario[(diario.kwh > 0) & (diario.cap_MW > 0)].copy()

primer = diario.groupby("code").date.min().rename("primer_dia")
diario = diario.merge(primer, on="code")
arranque_observado = diario.primer_dia > pd.Timestamp(START) + pd.Timedelta(days=7)
diario["dias_desde_arranque"] = np.where(arranque_observado, (diario.date - diario.primer_dia).dt.days, 10_000)

print("Filas planta-día con generación y capacidad:", len(diario))
print("Días con FC mayor a 1.05:", int((diario.fc > 1.05).sum()), "(se descartan)")
diario = diario[diario.fc <= 1.05]
print("Plantas con arranque observado dentro de la ventana:", int(diario.groupby("code").dias_desde_arranque.min().lt(10_000).sum()))

resumen = (diario.groupby("code")
           .agg(primer_dia=("date", "min"), dias=("date", "nunique"), cap_MW=("cap_MW", "median"), fc_medio=("fc", "mean"))
           .reset_index()
           .merge(solares[["Code", "Name"]], left_on="code", right_on="Code").drop(columns="Code"))
resumen.sort_values("cap_MW", ascending=False).round(3)

## 4. Emparejar con el dataset por nombre y capacidad

XM no entrega coordenadas y el clima las necesita. El dataset del paper sí las trae para algunas plantas. Primero se calculan los tres candidatos más parecidos de cada
planta en ese dataset. Luego se deja una tabla de decisiones hechas a mano, con el motivo de cada una. Siete plantas no aparecen en el dataset con un nombre reconocible,
y para ellas se usa la ficha pública de GEM, que es la misma fuente de la que sale el dataset. Cada una se contrastó con prensa o con el desarrollador.

Reglas para aceptar un emparejamiento:
- El nombre normalizado coincide de forma clara.
- La capacidad es igual o muy parecida. Dos casos tienen capacidad distinta y quedan anotados.
- Si hay dos filas de GEM con el mismo nombre, se verifica con una fuente externa cuál corresponde a la planta que genera.
- Si el nombre se parece pero la capacidad o el lugar no encajan, se descarta.
- Si las coordenadas de la fuente contradicen el lugar que dice la propia ficha, no se usan.

In [ ]:
gem = pd.read_csv(GEM_CSV, sep=";", decimal=",", encoding="utf-8-sig")
col = gem[gem.country.eq("Colombia")].reset_index(drop=True)

PALABRAS = r"\b(agpe|gd|autog|planta|parque|solar|solares|fv|fotovoltaic[oa]|photovoltaic|power|plant|farm|project|pv|park|colombia|de|del|la|el|los|las|i|ii|iii|iv|v|vi)\b"
def normaliza(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(PALABRAS, " ", s)
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s).split())

col["nombre_norm"] = col.plant_name.map(normaliza)
xm = resumen[["code", "Name", "cap_MW"]].copy()

def candidatos(nombre, k=3):
    clave = normaliza(nombre)
    sim = col.nombre_norm.map(lambda t: difflib.SequenceMatcher(None, clave, t).ratio())
    return col.assign(similitud=sim).sort_values("similitud", ascending=False).head(k)

filas = []
for _, r in xm.iterrows():
    for _, c in candidatos(r.Name).iterrows():
        filas.append({"code": r.code, "xm": r.Name, "xm_MW": round(r.cap_MW, 1), "gem": c.plant_name,
                      "gem_MW": c.capacity, "gem_estado": c.operational_status, "similitud": round(c.similitud, 2)})
pd.DataFrame(filas).head(30)

In [ ]:
# Clave: código XM. Valor: (nombre exacto en GEM, motivo)
EMPAREJAMIENTOS = {
    "3DDT": ("La Loma Solar Park",             "150 MW en ambos. 'Latam Solar' es el desarrollador"),
    "3HF5": ("Fundacion (Latamsolar) solar farm", "100 MW en ambos. El parque Fundación de Enel está en Pivijay, Magdalena (pv magazine LATAM, 2023; Enel Green Power, 2023). "
                                                "El geocodificador de Open-Meteo ubica Pivijay en 10.462, -74.616, igual que esta fila de GEM. "
                                                "La otra fila, 'Fundación Solar Park' (10.518, -74.182), cae en el municipio de Fundación, 48 km al este, y se descarta"),
    "3INX": ("Caracolí Solar Plant",           "50 MW en ambos. GEM la marca pre-construcción pero XM la tiene generando"),
    "3J2B": ("Planeta Rica Solar Park",        "19.9 MW en ambos"),
    "4RGG": ("Baranoa (Cecropia) solar farm",  "19.9 MW en ambos. GEM la marca pre-construcción"),
    "5F43": ("Alma Solar PV solar farm",       "9.8 MW en ambos. Hay otra 'Alma solar farm' de 20 MW en otra ubicación. La ubicación de esta fila no se contrastó con una fuente externa y la planta tiene solo unos 60 días de datos"),
    "EPFV": ("El Paso PV Solar Park",          "68 MW en XM, 70 MW en GEM"),
    "GYPO": ("Guayepo PV Solar Park",          "370 MW en XM, 486.7 MW en GEM, que suma más etapas. Mismo sitio. XM registra Guayepo III como otro recurso de 200 MW"),
    "MATA": ("La Mata solar farm (Colombia)",  "80 MW en ambos"),
    "TPUY": ("Tepuy solar farm",               "83 MW en ambos"),
}

# Plantas que el dataset del paper no trae con ese nombre. Las coordenadas salen de la ficha pública de GEM,
# la misma fuente que alimenta el dataset. Cada una se contrastó con prensa o con el desarrollador.
GEM_WIKI = "https://www.gem.wiki/"
EXTRAS = {
    "3EFY": {"gem": "Los Llanos Solar Forest, fase 4", "lat": 4.2152, "lon": -72.0266, "precision": "aproximada (GEM)",
             "fuente": GEM_WIKI + "Los_Llanos_Solar_Forest",
             "motivo": "Puerto Gaitán, Meta. 19.9 MWac en GEM y en XM. Fases 4 y 5 inauguradas en julio de 2022 (pv magazine LATAM, 2022)"},
    "3EDL": {"gem": "Los Llanos Solar Forest, fase 5", "lat": 4.2108, "lon": -72.0310, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Los_Llanos_Solar_Forest",
             "motivo": "Puerto Gaitán, Meta. 17.9 MWac en GEM y en XM"},
    "3K8C": {"gem": "Bolívar PV Solar Forest, fase 503", "lat": 10.5732, "lon": -74.8904, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Bolivar_PV_Solar_Forest",
             "motivo": "Sabanalarga, Atlántico. BSB significa Bosques Solares de Bolívar, de ISAGEN, cinco plantas de 19.9 MW (pv magazine LATAM, 2024-07-29)"},
    "3K8E": {"gem": "Bolívar PV Solar Forest, fase 504", "lat": 10.5796, "lon": -74.8903, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Bolivar_PV_Solar_Forest",
             "motivo": "Sabanalarga, Atlántico. Está a unos 0.7 km de la fase 503. Que la fase 503 sea 3K8C y la 504 sea 3K8E se asume por la numeración, sin otra fuente"},
    "3IZ6": {"gem": "La Unión Solar Park (Colombia)", "lat": 8.6920, "lon": -75.7962, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "La_Uni%C3%B3n_Solar_Park",
             "motivo": "Montería, Córdoba. Solarpack (hoy Zelestra). 99.9 MWac en GEM y 100 MW en XM. El código XM y la fecha de arranque no se contrastaron con una fuente externa"},
    "3IRX": {"gem": "Portón Del Sol PV Solar Park", "lat": 5.4104, "lon": -74.7141, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Port%C3%B3n_Del_Sol_PV_Solar_Park",
             "motivo": "La Dorada, Caldas. Enerfín. 102 MW en GEM y en XM. El Espectador dice que entró en operación comercial el 8 de marzo de 2024, la misma fecha de la primera generación en XM"},
    "3IQA": {"gem": "Sunnorte Solar Plant", "lat": 8.2247, "lon": -73.3233, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Sunnorte_Solar_Plant",
             "motivo": "Ocaña, Norte de Santander. Ecoener. 35 MWac en GEM y en XM. GEM y Ecoener dicen 2023, y XM muestra la primera generación en noviembre de 2024. La diferencia de fechas no se explicó"},
}
NO_EMPAREJADAS = {
    "3PWF": "Autogenerador de 4.99 MW. Una hipótesis (Celsia Solar Palmira 2) no se confirmó con ninguna fuente. Queda fuera",
    "4YCF": "Shangri-La, 160 MW. Departamento confirmado (Tolima) pero las coordenadas de GEM caen unos 230 km al este de Ibagué y contradicen su propia ficha. Tiene además pocos días de datos",
}

sin_emparejar = xm[~xm.code.isin(list(EMPAREJAMIENTOS) + list(EXTRAS))]
print(f"Emparejadas con el dataset: {len(EMPAREJAMIENTOS)} | con la ficha de GEM: {len(EXTRAS)} | de {len(xm)}")
print("Sin emparejar (sin coordenadas, quedan fuera por ahora):")
print(sin_emparejar.assign(cap_MW=lambda d: d.cap_MW.round(1)).to_string(index=False))

In [ ]:
plantas = []
for code_xm, (nombre_gem, motivo) in EMPAREJAMIENTOS.items():
    g = col[col.plant_name == nombre_gem]
    assert len(g) == 1, f"{nombre_gem}: {len(g)} filas en GEM"
    g = g.iloc[0]
    plantas.append({"code": code_xm, "planta": xm.set_index("code").loc[code_xm, "Name"],
                    "lat": g.latitude, "lon": g.longitude, "cap_gem_MW": g.capacity, "estado_gem": g.operational_status})
for code_xm, e in EXTRAS.items():
    plantas.append({"code": code_xm, "planta": xm.set_index("code").loc[code_xm, "Name"],
                    "lat": e["lat"], "lon": e["lon"], "cap_gem_MW": np.nan, "estado_gem": "operating (ficha GEM)"})
plantas = pd.DataFrame(plantas)

# Grupos por cercanía: plantas a menos de CLUSTER_KM comparten clima y no son observaciones independientes
def haversine_km(a, b):
    la1, lo1, la2, lo2 = map(np.radians, [a[0], a[1], b[0], b[1]])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))

D = np.array([[haversine_km(a, b) for b in plantas[["lat", "lon"]].values] for a in plantas[["lat", "lon"]].values])
plantas["grupo"] = fcluster(linkage(squareform(D, checks=False), method="single"), CLUSTER_KM, criterion="distance")
print("Grupos de plantas a menos de", CLUSTER_KM, "km (enlace simple):", plantas.grupo.nunique())
plantas.round(3)

In [ ]:
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(EXTRAS).T[["gem", "lat", "lon", "precision", "fuente", "motivo"]])
print("Plantas de XM investigadas y descartadas:")
for k, v in NO_EMPAREJADAS.items():
    print(f"  {k}: {v}")

## 5. Clima por hora en cada planta

Open-Meteo devuelve por hora la radiación de onda corta (horizontal, promedio de la hora anterior), la nubosidad total, la temperatura,
la humedad y el viento. Pedimos también la radiación en plano inclinado (`global_tilted_irradiance`) en dos variantes: inclinación fija de 10° orientada
al sur, y el modo `tilt=nan` que Open-Meteo documenta como seguidor horizontal. No verificamos que ese modo equivalga a un seguidor de un eje norte-sur que sigue al sol de este a oeste, así que su resultado no se interpreta. Los 10° son una suposición nuestra para un sitio cercano al ecuador. La tecnología
real de cada planta no la conocemos.

Se piden dos productos para las mismas coordenadas: `era5` y `best_match`. Open-Meteo guarda cada valor en la celda de su modelo, y la celda más cercana a una
planta cambia con el modelo. Esa diferencia sirve para ver cuánto depende el resultado de la fuente de radiación.

In [ ]:
BASE_VARS = "shortwave_radiation,cloud_cover,temperature_2m,relative_humidity_2m,wind_speed_10m"

def om_get(params, ruta):
    if ruta.exists():
        return pd.read_csv(ruta)
    ultimo_error = None
    for intento in range(6):
        try:
            r = requests.get(OM_URL, params=params, timeout=180)
            h = pd.DataFrame(r.json()["hourly"])
            break
        except (requests.RequestException, ValueError, KeyError) as e:
            ultimo_error = e
            time.sleep(10 * (intento + 1))      # Open-Meteo limita la tasa de consultas; esperamos y reintentamos
    else:
        raise RuntimeError(f"Open-Meteo no devolvió datos ({ruta.name}): {ultimo_error}")
    h.to_csv(ruta, index=False)
    time.sleep(5)
    return h

def clima_planta(code_xm, lat, lon):
    comunes = {"latitude": lat, "longitude": lon, "start_date": str(START), "end_date": str(END), "timezone": "America/Bogota"}
    out = {}
    for modelo in ["era5", "best_match"]:
        out[modelo] = om_get({**comunes, "hourly": BASE_VARS, "models": modelo}, DATA / f"clima_{code_xm}_{modelo}.csv")
    out["gti10"] = om_get({**comunes, "hourly": "global_tilted_irradiance", "tilt": 10, "azimuth": 0}, DATA / f"clima_{code_xm}_gti10.csv")
    out["gtitrk"] = om_get({**comunes, "hourly": "global_tilted_irradiance", "tilt": "nan", "azimuth": 0}, DATA / f"clima_{code_xm}_gtitrk.csv")
    return out

crudo = {p.code: clima_planta(p.code, p.lat, p.lon) for p in plantas.itertuples()}
# Celda del modelo que respondió para cada planta (se guarda para no repetir la consulta)
ruta_celdas = DATA / "celdas_modelo.json"
celdas = json.loads(ruta_celdas.read_text()) if ruta_celdas.exists() else {}
for p in plantas.itertuples():
    for modelo in ["era5", "best_match"]:
        clave = f"{p.code}_{modelo}"
        if clave in celdas:
            continue
        try:
            r = requests.get(OM_URL, params={"latitude": p.lat, "longitude": p.lon, "start_date": "2025-03-01", "end_date": "2025-03-01",
                                             "hourly": "shortwave_radiation", "models": modelo}, timeout=60).json()
            celdas[clave] = [r["latitude"], r["longitude"]]
        except (requests.RequestException, ValueError, KeyError):
            continue
        time.sleep(1)
ruta_celdas.write_text(json.dumps(celdas))

filas = []
for p in plantas.itertuples():
    f = {"planta": p.planta, "lat": round(p.lat, 3), "lon": round(p.lon, 3)}
    for modelo in ["era5", "best_match"]:
        c = celdas.get(f"{p.code}_{modelo}")
        f[f"km_a_celda_{modelo}"] = round(haversine_km((p.lat, p.lon), c), 1) if c else np.nan
    filas.append(f)
pd.DataFrame(filas)

### Resúmenes diarios

Para evitar problemas de alineación entre las horas de XM y las de Open-Meteo trabajamos con resúmenes diarios.
Al cruzar a nivel diario evitamos cualquier mínimo desfase horario entre ambas fuentes.

- Radiación diaria en kWh/m²: suma horaria dividida entre 1000.
- Nubosidad media en horas de luz (radiación mayor que 0).
- Temperatura, humedad y viento: medias diarias.
- Índice físico de energía: radiación horaria multiplicada por un factor de temperatura. La temperatura de celda sigue la regla de NOCT,
  `Tc = Ta + (NOCT - 20) / 800 · G`, con NOCT de 45 °C (Skoplaki y Palyvos, 2009, *Renewable Energy*). La pérdida es de 0.4 % por grado sobre 25 °C, un valor típico
  dentro del rango que revisan Skoplaki y Palyvos (2009, *Solar Energy*). Son supuestos genéricos, no los datos de cada planta.

In [ ]:
NOCT, GAMMA = 45.0, -0.004

def energia_fisica(g, ta):
    tc = ta + (NOCT - 20.0) / 800.0 * g
    return (g * (1 + GAMMA * (tc - 25.0))).clip(lower=0)

def resume(h, sufijo, extra_clima=True):
    h = h.copy()
    h["date"] = pd.to_datetime(h.time.str[:10])
    g = h.shortwave_radiation
    h["e_fis"] = energia_fisica(g, h.temperature_2m)
    luz = h.shortwave_radiation > 0
    out = h.groupby("date").agg(rad=("shortwave_radiation", lambda s: s.sum() / 1000),
                                e_fis=("e_fis", lambda s: s.sum() / 1000))
    out["nube"] = h[luz].groupby("date").cloud_cover.mean()
    if extra_clima:
        out["temp"] = h.groupby("date").temperature_2m.mean()
        out["humedad"] = h.groupby("date").relative_humidity_2m.mean()
        out["viento"] = h.groupby("date").wind_speed_10m.mean()
    return out.add_suffix(sufijo).reset_index()

def gti_diaria(h, nombre):
    h = h.copy(); h["date"] = pd.to_datetime(h.time.str[:10])
    return h.groupby("date").global_tilted_irradiance.sum().div(1000).rename(nombre).reset_index()

clima = []
for p in plantas.itertuples():
    c = crudo[p.code]
    t = (resume(c["best_match"], "_bm")
         .merge(resume(c["era5"], "_e5", extra_clima=False), on="date")
         .merge(gti_diaria(c["gti10"], "gti10"), on="date")
         .merge(gti_diaria(c["gtitrk"], "gtitrk"), on="date"))
    clima.append(t.assign(code=p.code))
clima = pd.concat(clima, ignore_index=True)
clima.describe().round(2).T[["count", "mean", "std", "min", "max"]]

## 6. Tabla diaria final

Una fila por planta y día con el FC real y el clima de ese día. Se guarda en `tabla_diaria_colombia.csv` para que otros cuadernos partan de ahí.
La tabla principal excluye los primeros `RAMP_DAYS` días de las plantas con arranque observado y las plantas que quedan con menos de `MIN_DIAS` días útiles. Las tablas sin esos filtros se conservan para la sensibilidad de la sección 9.4.

In [ ]:
completa = (diario.merge(plantas[["code", "planta", "lat", "lon", "grupo"]], on="code")
                  .merge(clima, on=["code", "date"], how="inner")
                  .dropna(subset=["rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm", "gti10", "gtitrk"])
                  .sort_values(["code", "date"]).reset_index(drop=True))
completa["fc_ayer"] = completa.groupby("code").fc.shift(1)
completa.loc[completa.date.diff().dt.days.ne(1) | completa.code.ne(completa.code.shift(1)), "fc_ayer"] = np.nan

tabla_base = completa[completa.dias_desde_arranque >= RAMP_DAYS].reset_index(drop=True)
dias_utiles = tabla_base.groupby("code").size()
validas = dias_utiles[dias_utiles >= MIN_DIAS].index
print("Plantas con menos de", MIN_DIAS, "días útiles tras excluir el arranque:", dias_utiles[dias_utiles < MIN_DIAS].to_dict())
tabla = tabla_base[tabla_base.code.isin(validas)].reset_index(drop=True)
tabla.to_csv("tabla_diaria_colombia.csv", index=False)

print(f"Filas (principal): {len(tabla)} | plantas: {tabla.code.nunique()} | grupos de cercanía: {tabla.grupo.nunique()}")
print(f"Filas excluidas por puesta en marcha: {len(completa) - len(tabla_base)}")
print(tabla.groupby("planta").agg(dias=("fc", "size"), fc_medio=("fc", "mean")).round(3).to_string())

## 6b. EDA de la base de datos

### Diccionario de variables de `tabla_diaria_colombia.csv`

Una fila es una planta en un día.

| Columna | Qué es | Unidad | Fuente |
|---|---|---|---|
| `code` | Código del recurso en XM | texto | XM |
| `planta` | Nombre del recurso en XM | texto | XM |
| `date` | Día, hora local de Bogotá | fecha | XM |
| `kwh` | Energía generada en el día (suma de las 24 horas) | kWh | XM, métrica `Gene` |
| `cap_MW` | Capacidad efectiva neta de ese día | MW | XM, métrica `CapEfecNeta` |
| `fc` | Factor de capacidad: `kwh / (cap_MW · 1000 · 24)`. Es la variable que se predice | fracción de 0 a 1 | calculada |
| `fc_ayer` | `fc` del día anterior, vacío si el día anterior falta | fracción | calculada |
| `primer_dia` | Primer día con generación y capacidad en la ventana | fecha | calculada |
| `dias_desde_arranque` | Días desde `primer_dia`. Vale 10000 si la planta ya operaba al inicio de la ventana | días | calculada |
| `lat`, `lon` | Coordenadas de la planta | grados | dataset del paper o ficha de GEM (sección 4) |
| `grupo` | Zona de plantas a menos de 100 km entre sí | entero | calculada |
| `rad_bm` | Radiación global horizontal del día, modelo `best_match` | kWh/m²/día | Open-Meteo |
| `rad_e5` | Lo mismo con el modelo `era5` | kWh/m²/día | Open-Meteo |
| `e_fis_bm`, `e_fis_e5` | Radiación horaria con la pérdida por temperatura de celda (NOCT 45 °C, 0.4 % por °C), sumada en el día | kWh/m²/día | calculada |
| `nube_bm`, `nube_e5` | Nubosidad total media en las horas de luz | % | Open-Meteo |
| `temp_bm` | Temperatura media del aire a 2 m | °C | Open-Meteo |
| `humedad_bm` | Humedad relativa media a 2 m | % | Open-Meteo |
| `viento_bm` | Velocidad media del viento a 10 m | km/h | Open-Meteo |
| `gti10` | Radiación en un plano fijo de 10° orientado al sur | kWh/m²/día | Open-Meteo |
| `gtitrk` | Radiación con el modo `tilt=nan` de Open-Meteo (no interpretado, ver sección 5) | kWh/m²/día | Open-Meteo |

La tecnología de cada planta (seguidor o estructura fija, potencia pico DC frente a capacidad AC) no está en la base. Queda pendiente: es lo que más
probablemente explica por qué unas plantas tienen un nivel de FC más alto que otras.

In [ ]:
print("Dimensiones:", tabla.shape, "| plantas:", tabla.code.nunique(), "| zonas:", tabla.grupo.nunique(),
      "| del", tabla.date.min().date(), "al", tabla.date.max().date())
VARS_EDA = ["fc", "kwh", "cap_MW", "rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
tabla[VARS_EDA].describe().T.round(3)

### Valores faltantes

Para cada planta se arma la rejilla de todos los días entre su primer día con datos y el 28 de febrero de 2026, y se cuenta qué falta en cada día.
Las horas vacías dentro de un día en XM son horas de noche sin generación, así que no cuentan como faltantes. Un día cuenta como faltante en generación
si XM no trae registro o trae 0 kWh.

In [ ]:
plantas_ok = tabla.code.unique()
filas = []
for code in plantas_ok:
    ini = diario.loc[diario.code == code, "date"].min()
    for d in pd.date_range(ini, pd.Timestamp(END)):
        filas.append((code, d))
rejilla = pd.DataFrame(filas, columns=["code", "date"])
g_dia = gen.groupby(["code", "date"]).kwh.sum().reset_index()
rejilla = (rejilla.merge(g_dia, on=["code", "date"], how="left")
                  .merge(cap_dia, on=["code", "date"], how="left")
                  .merge(clima[["code", "date", "rad_bm", "rad_e5", "nube_bm"]], on=["code", "date"], how="left"))

falt = pd.DataFrame({
    "dias_esperados": rejilla.groupby("code").size(),
    "sin_generacion": rejilla.assign(x=rejilla.kwh.isna() | (rejilla.kwh <= 0)).groupby("code").x.sum(),
    "sin_capacidad": rejilla.assign(x=rejilla.cap_MW.isna() | (rejilla.cap_MW <= 0)).groupby("code").x.sum(),
    "sin_clima": rejilla.assign(x=rejilla.rad_bm.isna() | rejilla.rad_e5.isna() | rejilla.nube_bm.isna()).groupby("code").x.sum(),
    "dias_en_tabla": tabla.groupby("code").size(),
}).join(tabla.groupby("code").planta.first())
falt["pct_faltante"] = 100 * (1 - (falt.dias_en_tabla + 0) / falt.dias_esperados)
print("Faltantes por columna en la tabla final:", int(tabla.isna().sum().drop("fc_ayer").sum()), "valores vacíos fuera de fc_ayer")
print("fc_ayer vacío:", int(tabla.fc_ayer.isna().sum()), "filas (primer día de cada planta o día anterior ausente)")
falt.set_index("planta").sort_values("pct_faltante", ascending=False).round(1)

La columna `pct_faltante` mezcla dos causas: días que faltan en XM y los días de puesta en marcha que se excluyen a propósito. Las plantas con arranque observado
pierden sus primeros 60 días por diseño, y eso no es un dato faltante.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), gridspec_kw={"width_ratios": [1.3, 1.2, 1]})

orden = tabla.groupby("planta").fc.median().sort_values().index
axes[0].boxplot([tabla.loc[tabla.planta == p, "fc"] for p in orden], vert=False, showfliers=False)
axes[0].set_yticks(range(1, len(orden) + 1)); axes[0].set_yticklabels([p[:24] for p in orden], fontsize=8)
axes[0].set_xlabel("factor de capacidad diario"); axes[0].set_title("Distribución del FC por planta")

cob = (tabla.assign(mes=tabla.date.dt.to_period("M").astype(str)).groupby(["planta", "mes"]).size().unstack(fill_value=0))
im = axes[1].imshow(cob.values, aspect="auto", cmap="Greens", vmin=0, vmax=31)
axes[1].set_yticks(range(len(cob.index))); axes[1].set_yticklabels([p[:24] for p in cob.index], fontsize=8)
axes[1].set_xticks(range(0, len(cob.columns), 3)); axes[1].set_xticklabels(cob.columns[::3], rotation=90, fontsize=8)
axes[1].set_title("Días con datos por planta y mes"); fig.colorbar(im, ax=axes[1], shrink=.7)

VARS_CORR = ["fc", "rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
cen = tabla[VARS_CORR + ["code"]].copy()
for v in VARS_CORR:
    cen[v] = cen[v] - cen.groupby("code")[v].transform("mean")
cor = cen[VARS_CORR].corr(method="spearman")
im2 = axes[2].imshow(cor.values, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_xticks(range(len(VARS_CORR))); axes[2].set_xticklabels(VARS_CORR, rotation=90)
axes[2].set_yticks(range(len(VARS_CORR))); axes[2].set_yticklabels(VARS_CORR)
for i in range(len(VARS_CORR)):
    for j in range(len(VARS_CORR)):
        axes[2].text(j, i, f"{cor.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
axes[2].set_title("Spearman dentro de planta (variables centradas)")
fig.tight_layout(); fig.savefig("fig_eda_base_colombia.png", dpi=130, bbox_inches="tight"); plt.show()

La matriz de la derecha usa las variables con la media de cada planta restada, así que mide cómo se mueven juntas de un día a otro. La radiación
y la nubosidad están muy correlacionadas, igual que la temperatura y la humedad con la radiación. Eso explica por qué añadir esas variables a un modelo
que ya tiene la radiación aporta poco.

In [ ]:
meta = plantas[plantas.code.isin(tabla.code.unique())][["code", "planta", "lat", "lon", "grupo"]].copy()
meta["fuente_coordenadas"] = meta.code.map(lambda c: "dataset del paper: " + EMPAREJAMIENTOS[c][0] if c in EMPAREJAMIENTOS else "ficha GEM: " + EXTRAS[c]["fuente"])
meta["precision"] = meta.code.map(lambda c: EXTRAS[c]["precision"] if c in EXTRAS else "la del dataset (GEM)")
meta["motivo"] = meta.code.map(lambda c: EMPAREJAMIENTOS[c][1] if c in EMPAREJAMIENTOS else EXTRAS[c]["motivo"])
# Lo poco que se sabe de tecnología, con su fuente. Todo lo demás queda pendiente.
DC_CONOCIDO = {
    "3EFY": ("27.39", "https://www.pv-magazine-latam.com/2022/07/15/inauguran-en-colombia-las-fases-4-y-5-de-bosques-de-los-llanos-complejo-solar-que-completa-125-mwp/"),
    "3EDL": ("25.01", "https://www.pv-magazine-latam.com/2022/07/15/inauguran-en-colombia-las-fases-4-y-5-de-bosques-de-los-llanos-complejo-solar-que-completa-125-mwp/"),
    "3IQA": ("41", "https://www.ecoener.es/proyectos/planta-fotovoltaica-sunnorte"),
}
meta["dc_MWp"] = meta.code.map(lambda c: DC_CONOCIDO.get(c, ("pendiente", ""))[0])
meta["fuente_dc"] = meta.code.map(lambda c: DC_CONOCIDO.get(c, ("", ""))[1])
meta["montaje"] = "pendiente"
meta = meta.merge(tabla.groupby("code").agg(dias=("fc", "size"), fc_medio=("fc", "mean"), cap_MW=("cap_MW", "median")).reset_index(), on="code")
meta.to_csv("plantas_colombia.csv", index=False)
meta[["code", "planta", "lat", "lon", "grupo", "dias", "fc_medio", "cap_MW", "precision"]].round(3)

### Tecnología de cada planta: pendiente

La tabla siguiente muestra lo que falta. Ninguna planta tiene confirmado el tipo de montaje (seguidor de un eje o estructura fija). Solo tres tienen una potencia
pico DC citada por una fuente, que reportó un investigador y no se revisó a mano. Mientras esto no se complete, la diferencia de nivel entre plantas
(el 16 % de la varianza del FC) queda sin explicar.

In [ ]:
tec = meta[["code", "planta", "cap_MW", "dc_MWp", "montaje", "fuente_dc"]].copy()
tec["dc_ac"] = pd.to_numeric(tec.dc_MWp, errors="coerce") / tec.cap_MW
print("Plantas sin montaje confirmado:", int((tec.montaje == "pendiente").sum()), "de", len(tec))
print("Plantas sin potencia DC:", int((tec.dc_MWp == "pendiente").sum()), "de", len(tec))
tec.round(2)

## 7. Qué se ve antes de modelar

Una gráfica por planta del FC contra la radiación del día, coloreada por nubosidad. Si el clima explica la generación, los puntos suben con la radiación.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
for ax, (nombre, g) in zip(axes.ravel(), tabla.groupby("planta")):
    sc = ax.scatter(g.rad_bm, g.fc, c=g.nube_bm, s=6, cmap="viridis_r", alpha=.7)
    ax.set_title(f"{nombre[:22]}  rho={spearmanr(g.rad_bm, g.fc)[0]:.2f}", fontsize=9)
    ax.set_xlabel("radiación (kWh/m²/día)")
axes[0, 0].set_ylabel("factor de capacidad"); axes[1, 0].set_ylabel("factor de capacidad")
fig.colorbar(sc, ax=axes, label="nubosidad media de día (%)", shrink=.8)
fig.suptitle("Factor de capacidad real contra radiación modelada (best_match), por planta", y=.98)
fig.savefig("fig_fc_vs_radiacion.png", dpi=130, bbox_inches="tight")
plt.show()

cols = ["rad_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
(tabla.groupby("planta").apply(lambda g: pd.Series({v: spearmanr(g[v], g.fc)[0] for v in cols})).round(2))

La temperatura sale con correlación positiva con el FC en todas las plantas. Físicamente el calor reduce el rendimiento, así que esa
correlación viene de que los días soleados también son más cálidos. No se debe leer como un efecto de la temperatura.

## 8. Cómo se evalúa

Hay tres decisiones de diseño que vienen de la revisión metodológica de este cuaderno.

Nivel de cada planta. Cada planta tiene su propio nivel de FC por factores que no medimos (seguidores, relación DC/AC, edad). La varianza entre plantas se calcula al empezar la sección 9.1.
Para medir solo lo que el clima puede explicar, a cada variable se le resta la media de su planta y se predice el sube y baja diario. Eso presupone conocer el nivel de la planta.
En la evaluación temporal el nivel se calcula solo con la primera mitad de los días de cada planta.

Plantas que no se pueden tratar como independientes. Varias plantas están a menos de 100 km y comparten clima el mismo día. La validación deja fuera un grupo
completo de plantas cercanas a la vez (`LeaveOneGroupOut` sobre `grupo`), no una planta suelta.

Incertidumbre. Con 16 plantas en 6 zonas, la unidad de muestreo es la planta. Los intervalos salen de remuestrear plantas (Field y Welsh, 2007), no filas.
Las comparaciones entre modelos usan diferencias pareadas por planta con la prueba de Wilcoxon. Con 16 pares la prueba tiene poca potencia, así que todo es exploratorio.

Métricas. R² dentro de la planta, error absoluto medio relativo al FC medio de esa planta (MAE relativo) y la mejora frente a dos referencias sin clima:
predecir siempre el nivel de la planta y repetir el FC del día anterior.

In [ ]:
FEATS = {
    "A. radiación era5":                        ["rad_e5"],
    "B. radiación best_match":                  ["rad_bm"],
    "C. física: radiación con temperatura":     ["e_fis_bm"],
    "D. plano con tilt=nan (modo seguidor de Open-Meteo)": ["gtitrk"],
    "E. radiación en plano fijo de 10°":        ["gti10"],
    "F. radiación + nubosidad":                 ["rad_bm", "nube_bm"],
    "G. todas (rad, nube, temp, humedad, viento)": ["rad_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"],
    "H. solo nubosidad":                        ["nube_bm"],
}

def centrar(df, cols):
    c = df.copy()
    for v in cols:
        c[v + "_c"] = c[v] - c.groupby("code")[v].transform("mean")
    return c

def metricas_planta(y_real, y_pred, y_media, y_ayer):
    sst = ((y_real - y_real.mean()) ** 2).sum()
    mae = np.abs(y_real - y_pred).mean()
    mae_nivel = np.abs(y_real - y_media).mean()
    ok = ~np.isnan(y_ayer)
    mae_pers = np.abs(y_real[ok] - y_ayer[ok]).mean() if ok.sum() > 5 else np.nan
    return {"r2": 1 - ((y_real - y_pred) ** 2).sum() / sst, "mae_rel": mae / y_media,
            "mejora_vs_nivel": 1 - mae / mae_nivel,
            "mejora_vs_persistencia": 1 - np.abs(y_real[ok] - y_pred[ok]).mean() / mae_pers if ok.sum() > 5 else np.nan}

def evalua_nivel_conocido(df, feats, modelo=LinearRegression):
    cols = feats + ["fc"]
    c = centrar(df, cols)
    filas = []
    for tr, te in LeaveOneGroupOut().split(c, groups=c.grupo):
        m = modelo().fit(c.iloc[tr][[v + "_c" for v in feats]], c.iloc[tr].fc_c)
        d = c.iloc[te].copy()
        d["pred_c"] = m.predict(d[[v + "_c" for v in feats]])
        for code, g in d.groupby("code"):
            media = df[df.code == code].fc.mean()
            y = g.fc.values; pred = g.pred_c.values + media
            filas.append({"code": code, "planta": g.planta.iloc[0], "n": len(g),
                          **metricas_planta(y, pred, media, g.fc_ayer.values)})
    return pd.DataFrame(filas)

def evalua_temporal(df, feats, modelo=LinearRegression):
    d = df.copy()
    d["mitad"] = d.groupby("code").cumcount() >= d.groupby("code").fc.transform("size") // 2
    ent, prueba = d[~d.mitad], d[d.mitad]
    medias = ent.groupby("code")[feats + ["fc"]].mean()
    ent_c = ent.copy(); prueba_c = prueba.copy()
    for v in feats + ["fc"]:
        ent_c[v + "_c"] = ent[v] - ent.code.map(medias[v]); prueba_c[v + "_c"] = prueba[v] - prueba.code.map(medias[v])
    filas = []
    for g_id in d.grupo.unique():
        tr, te = ent_c[ent_c.grupo != g_id], prueba_c[prueba_c.grupo == g_id]
        m = modelo().fit(tr[[v + "_c" for v in feats]], tr.fc_c)
        for code, g in te.groupby("code"):
            media = medias.loc[code, "fc"]
            pred = m.predict(g[[v + "_c" for v in feats]]) + media
            filas.append({"code": code, "planta": g.planta.iloc[0], "n": len(g), **metricas_planta(g.fc.values, pred, media, g.fc_ayer.values)})
    return pd.DataFrame(filas)

def ic_bootstrap(valores, B=5000):
    v = np.asarray(valores); n = len(v)
    med = [np.median(v[RNG.integers(0, n, n)]) for _ in range(B)]
    return np.percentile(med, [2.5, 97.5])

def tabla_resumen(funcion, conjuntos=FEATS, df=None):
    df = tabla if df is None else df
    filas, por_planta = [], {}
    for nombre, feats in conjuntos.items():
        r = funcion(df, feats)
        por_planta[nombre] = r.set_index("code")
        lo, hi = ic_bootstrap(r.r2)
        filas.append({"variables": nombre, "r2_mediana": r.r2.median(), "ic95_bajo": lo, "ic95_alto": hi,
                      "r2_min": r.r2.min(), "r2_max": r.r2.max(), "mae_rel_medio": r.mae_rel.mean(),
                      "mejora_vs_nivel": r.mejora_vs_nivel.mean(), "mejora_vs_persistencia": r.mejora_vs_persistencia.mean()})
    return pd.DataFrame(filas).set_index("variables"), por_planta

## 9. Resultados

### 9.1 Nivel de la planta conocido

El nivel de cada planta se calcula con todos sus días, y la validación deja fuera un grupo completo de plantas cercanas.

In [ ]:
w = tabla.groupby("code").fc.transform(lambda s: s - s.mean()).var()
print(f"Varianza del FC entre plantas: {100 * (1 - w / tabla.fc.var()):.0f} %  |  dentro de cada planta, día a día: {100 * w / tabla.fc.var():.0f} %")

COLS_1 = ["r2_mediana", "ic95_bajo", "ic95_alto", "r2_min", "r2_max"]
COLS_2 = ["mae_rel_medio", "mejora_vs_nivel", "mejora_vs_persistencia"]
res_A, pp_A = tabla_resumen(evalua_nivel_conocido)
display(res_A[COLS_1].round(3)); display(res_A[COLS_2].round(3))

In [ ]:
c = centrar(tabla, ["rad_bm", "fc"])
pend = np.polyfit(c.rad_bm_c, c.fc_c, 1)[0]
print(f"Pendiente dentro de planta: {100 * pend:.1f} puntos de FC por cada kWh/m² de radiación diaria")
print(f"Proporcionalidad simple: FC medio / radiación media = {100 * tabla.fc.mean() / tabla.rad_bm.mean():.1f} puntos por kWh/m²")
print(f"Plantas: {tabla.code.nunique()} | grupos de cercanía: {tabla.grupo.nunique()} | filas: {len(tabla)} | del {tabla.date.min().date()} al {tabla.date.max().date()}")

In [ ]:
base = "B. radiación best_match"
filas = []
for nombre, r in pp_A.items():
    if nombre == base:
        continue
    d = (r.r2 - pp_A[base].r2).dropna()
    p = wilcoxon(d).pvalue if (d != 0).any() else np.nan
    filas.append({"modelo": nombre, "dif_r2_media_vs_B": d.mean(), "dif_min": d.min(), "dif_max": d.max(), "p_wilcoxon": p})
pd.DataFrame(filas).set_index("modelo").round(3)

Los valores p son crudos, de una prueba con 16 pares y 7 comparaciones. No se corrigen por comparaciones múltiples porque no hay una hipótesis confirmatoria: se leen como orden de magnitud de la diferencia.

### 9.2 Nivel estimado con el pasado

Aquí el nivel de cada planta sale solo de la primera mitad de sus días, y se evalúa en la segunda mitad. Es una prueba más dura: las plantas cuyo nivel se desplaza en el tiempo salen peor.

In [ ]:
res_B, pp_B = tabla_resumen(evalua_temporal)
display(res_B[COLS_1].round(3)); display(res_B[COLS_2].round(3))

In [ ]:
pd.DataFrame({"r2 nivel conocido (B)": pp_A[base].r2, "r2 temporal (B)": pp_B[base].r2,
              "r2 nivel conocido (C física)": pp_A["C. física: radiación con temperatura"].r2,
              "r2 temporal (C física)": pp_B["C. física: radiación con temperatura"].r2}
             ).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

### 9.3 Un modelo flexible

Para comprobar si un modelo no lineal cambia algo, se prueba HistGradientBoosting (Friedman, 2001) con todas las variables centradas, con la misma validación por grupos.
Sin hyperparameter tuning y con 16 plantas, una pérdida frente al modelo lineal puede venir del sobreajuste del propio modelo, no del clima.

In [ ]:
r_hgb = evalua_nivel_conocido(tabla, FEATS["G. todas (rad, nube, temp, humedad, viento)"],
                              modelo=lambda: HistGradientBoostingRegressor(random_state=SEED))
lo, hi = ic_bootstrap(r_hgb.r2)
print(f"HGB, todas las variables centradas: R² mediana {r_hgb.r2.median():.3f}  (IC95 {lo:.3f} a {hi:.3f})")
print(f"Lineal, mismas variables:           R² mediana {res_A.loc['G. todas (rad, nube, temp, humedad, viento)', 'r2_mediana']:.3f}")

### 9.4 Sensibilidad a decisiones previas

Cada fila cambia una decisión y repite la evaluación con solo radiación (B) y con la física (C).

In [ ]:
def sens(df, etiqueta):
    cj = {k: FEATS[k] for k in ["B. radiación best_match", "C. física: radiación con temperatura"]}
    r, _ = tabla_resumen(evalua_nivel_conocido, cj, df)
    return r[["r2_mediana", "ic95_bajo", "ic95_alto"]].assign(escenario=etiqueta, n_filas=len(df)).reset_index()

esc = [
    sens(tabla, "principal (60 días de arranque excluidos)"),
    sens(completa[completa.code.isin(validas)], "sin excluir días de arranque"),
    sens(completa[completa.code.isin(validas) & (completa.dias_desde_arranque >= 90)], "90 días de arranque excluidos"),
    sens(tabla_base, "incluyendo ALMA II (8 días útiles)"),
    sens(tabla[tabla.fc >= 0.02], "sin días con FC < 0.02 (filtra por la respuesta, solo sensibilidad)"),
]
pd.concat(esc).set_index(["escenario", "variables"]).round(3)

### 9.5 Cuánto pesa la fuente de radiación

Los modelos A y B usan la misma regresión sobre la radiación de dos productos de Open-Meteo. Si la diferencia entre ellos es grande, la calidad del modelo de clima
es parte del techo del resultado.

In [ ]:
pd.DataFrame({"era5 (0.25°)": pp_A["A. radiación era5"].r2, "best_match": pp_A["B. radiación best_match"].r2}
            ).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

### 9.6 Modelos del curso, versión preliminar

Prueba de los modelos base que pide el curso sobre esta base: regresión lineal, Ridge, Lasso, KNN y SVR con kernel RBF, más HistGradientBoosting
como modelo flexible y la referencia de predecir el nivel de la planta.

Cómo se evita la fuga de información:
- La partición va primero. La validación deja fuera una zona completa de plantas cercanas (`LeaveOneGroupOut` sobre `grupo`).
- El nivel de cada planta se calcula solo con la primera mitad de sus días (el pasado) y se evalúa en la segunda mitad. Las plantas de la zona de prueba no entran al entrenamiento.
- El escalado (`StandardScaler`) va dentro de un `Pipeline`, así que se ajusta solo con los datos de entrenamiento de cada partición.
- Los hiperparámetros quedan fijos en valores por defecto razonables. No se ajustan aquí, porque ajustarlos con los mismos datos de prueba inflaría el resultado. El ajuste va en la versión final, con validación anidada por grupos.

Variables: radiación, nubosidad, temperatura, humedad y viento (`best_match`).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

class NivelPlanta:
    # Predice desviación 0: el FC es el nivel de la planta calculado con el pasado
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X))

MODELOS_CURSO = {
    "nivel de la planta (referencia)": lambda: NivelPlanta(),
    "regresión lineal": lambda: make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha=1)": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (alpha=0.001)": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.001, max_iter=10000)),
    "KNN (k=25)": lambda: make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25)),
    "SVR RBF (C=1, epsilon=0.01)": lambda: make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.01)),
    "HistGradientBoosting": lambda: HistGradientBoostingRegressor(random_state=SEED),
}
VARS_G = FEATS["G. todas (rad, nube, temp, humedad, viento)"]
filas, por_planta_curso = [], {}
for nombre, fab in MODELOS_CURSO.items():
    t0 = time.time()
    r = evalua_temporal(tabla, VARS_G, modelo=fab)
    por_planta_curso[nombre] = r.set_index("code").r2
    lo, hi = ic_bootstrap(r.r2)
    filas.append({"modelo": nombre, "r2_mediana": r.r2.median(), "ic95_bajo": lo, "ic95_alto": hi,
                  "mae_rel_medio": r.mae_rel.mean(), "mejora_vs_persistencia": r.mejora_vs_persistencia.mean(),
                  "segundos": time.time() - t0})
res_curso = pd.DataFrame(filas).set_index("modelo")
res_curso.round(3)

In [ ]:
pd.DataFrame(por_planta_curso).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

## 10. Modelos propuestos para la versión final

| Modelo | Por qué |
|---|---|
| Referencia física (radiación por un factor propio de cada planta) | Es lo que predice la física. Cualquier modelo de aprendizaje debe superarla para justificar su complejidad |
| Ridge y Lasso | Lineales, interpretables y con regularización. La relación con la radiación es casi lineal (sección 9.1), y Lasso muestra qué variables sobran |
| KNN y SVR con kernel | Modelos base del curso. Capturan no linealidades sin suponer una forma, aunque con pocas zonas pueden sobreajustar |
| HistGradientBoosting o XGBoost | Interacciones entre variables. En la prueba preliminar no superaron al lineal, así que entran como control |

Diseño para la versión final: los mismos grupos de zonas, el nivel de cada planta estimado con el pasado, y la búsqueda de hiperparámetros dentro de un
bucle interno que también agrupe por zona (validación anidada), con el escalado dentro del `Pipeline`. Lo que más cambiaría los resultados no es el
modelo sino los datos: la tecnología de cada planta, datos horarios y radiación medida.

## 11. Conclusiones

Muestra: 16 plantas con coordenadas verificadas y al menos 100 días útiles, en 6 zonas de cercanía, del 1 de enero de 2024 al 28 de febrero de 2026, con 8,589 filas planta-día y sin valores vacíos en la tabla final. Nueve plantas salen del dataset del paper y siete de la ficha pública de GEM, cada una contrastada con prensa o con el desarrollador. ALMA II sale por quedar con 8 días útiles, Palmira II porque su ubicación no se confirmó, y Shangri-La porque sus coordenadas en GEM contradicen su propia ficha.

Lo que muestran los resultados con el nivel de cada planta conocido (sección 9.1):
- La radiación diaria de Open-Meteo (`best_match`) explica una mediana de R² de 0.49 de la variación día a día dentro de una planta (IC95 de 0.38 a 0.51 remuestreando plantas). Por planta va de 0.09 (Sunnorte, en Ocaña) a 0.59 (Caracolí). Esa variación diaria es el 84 % de la varianza total del FC.
- Cada kWh/m² adicional de radiación diaria suma unos 4.7 puntos de factor de capacidad, casi lo mismo que sale de dividir el FC medio entre la radiación media (4.6). El resultado es el que predice la física, y lo que aporta este cuaderno es medirlo en 16 plantas colombianas reales.
- Frente a predecir el nivel de la planta, el error absoluto baja un 28 %. Frente a repetir el FC de ayer, baja un 31 %.
- La pérdida por temperatura de celda (0.4 % por °C, NOCT de 45 °C) no mejora la radiación sola (0.487 frente a 0.493). Los parámetros son genéricos.
- Añadir nubosidad, temperatura, humedad y viento sube el R² por planta en promedio 0.018 (de -0.012 a +0.036). Es un efecto pequeño. La nubosidad sola explica 0.17.
- La fuente de radiación pesa: con `era5` la mediana es 0.41 y con `best_match` 0.49. `best_match` gana en 13 de las 16 plantas.

Modelos del curso con el nivel calculado con el pasado y el escalado dentro de cada partición (sección 9.6): regresión lineal, Ridge y Lasso dan una mediana de 0.47. KNN da 0.40, SVR 0.36 y HistGradientBoosting 0.38. Los modelos flexibles no superan al lineal con 16 plantas y 6 zonas. La planta La Loma sale negativa en esta prueba (-0.47), porque su nivel se desplazó entre la primera y la segunda mitad del período. No verificamos la causa.

Sensibilidad (sección 9.4): la mediana va de 0.486 a 0.502 al cambiar la exclusión del arranque o incluir ALMA II.

Lo que estos resultados no permiten decir:
- Que la nubosidad explique algo además de la radiación. La radiación del modelo ya integra sus propias nubes, y la nubosidad diaria no capta cuánto cambia el cielo durante el día.
- Que 0.5 sea el techo del fenómeno. Cambiar de producto de radiación movió la mediana de 0.41 a 0.49, y la tecnología de cada planta no está medida.
- Que valga para cualquier zona. Hay 6 zonas, cinco en el norte y el centro del país y una en los Llanos. Sunnorte, en zona de montaña, sale mucho peor que el resto.

Pendiente:
1. La tecnología de cada planta (seguidor o fija, potencia DC frente a AC). Ninguna de las 16 tiene el montaje confirmado y solo 3 tienen la potencia DC con fuente.
2. Comparar la radiación modelada con radiación medida (estaciones del IDEAM) o satelital.
3. Repetir con datos horarios y con un índice de claridad para la pregunta de las nubes.
4. hyperparameter tuning con validación anidada por zonas, para la versión final.

# Resumen del Proyecto

Este apartado consolida todo el trabajo, desde la recolección de datos hasta el benchmarking y los siete experimentos. Las cifras corresponden a la ejecución de octubre de 2026.

| Cuaderno | Qué hace |
|---|---|
| `EDA_Corregido(Entregable 3).ipynb` | Limpia y explora la base mundial y genera `dataset_eda_corregido.csv`. |
| `Benchmark_Modelos_Base.ipynb` | Corre los modelos base sobre la base mundial. |
| `Clima_a_Generacion_Colombia.ipynb` | Construye la base de generación real de Colombia y prueba modelos preliminares. |
| `Benchmark_Colombia.ipynb` | Corre los modelos base sobre los datos de Colombia. |
| `Experimentos_1_a_7.ipynb` | Detallan el diseño, clasificación, regresión, optimizadores, rendimiento computacional, pruebas estadísticas e interpretabilidad. |
| `Resumen_Final.ipynb` | Cuaderno de síntesis con la ejecución completa y figuras. |

## 1. La pregunta original y cómo evolucionó

Arrancamos queriendo predecir la aptitud de un sitio para albergar una planta solar usando datos climáticos globales. Tomamos el dataset de Mantilla-Guerra et al. (2026), que trae un índice de aptitud solar (`solar_aptitude`). Al revisar la fórmula del paper, nos dimos cuenta de que el índice se calcula solo con variables de terreno (pendiente, orientación, sombreado y curvatura). El clima no influye. Esto nos obligó a dividir el proyecto en dos caminos:
1. Modelar el índice topográfico del dataset global para evaluar su generalización espacial.
2. Construir una base propia con generación real de plantas colombianas para ver cuánto influye realmente el clima.

## 2. Los datos recolectados

Juntamos información de cinco fuentes: el dataset base de Mantilla-Guerra (con potencial solar y datos topográficos), el Global Energy Monitor (para coordenadas y nombres), XM (el operador eléctrico en Colombia, para tener la generación real hora a hora), y Open-Meteo (para la radiación, nubosidad, viento, temperatura y humedad).

El proceso de limpieza fue riguroso. En la base mundial quitamos filas rotas, reparamos más de mil duplicados y resolvimos vacíos que eran sistemáticos. Esto nos dejó con 57,976 plantas y 15 predictoras. Para Colombia, armamos una serie con 8,589 días-planta provenientes de 16 plantas con coordenadas verificadas. 

## 3. El problema oculto del índice global

El hallazgo central del EDA es que el índice casi no correlaciona con la pendiente, a pesar de que esta es el 40% de su fórmula. En cambio, tiene una correlación de +0.58 con la longitud. Vimos que un modelo entrenado en dos continentes tiene un R² negativo cuando predice en un tercer continente. Esto indica que el índice captura principalmente el bloque o lote de procesamiento (por ejemplo, el bloque europeo) en lugar de una regla topográfica universal.

Por esta razón, tuvimos que evaluar todo usando bloques espaciales de 5° × 5°. Partir los datos al azar mezclaba plantas vecinas y daba métricas muy optimistas. 

## 4. Diseño experimental

Con la advertencia espacial en mente, ajustamos todos los hiperparámetros usando validación cruzada anidada: un bucle externo por bloques geográficos, y un bucle interno para buscar hiperparámetros. 

Comparamos modelos lineales (Logística, Ridge, Lasso) contra métodos más flexibles (KNN, SVM, y luego Random Forest y XGBoost). Frente al desbalance de clases (Baja tiene solo el 3%), probamos dar pesos en la función de costo, SMOTE y ADASYN. Evaluamos optimizadores como búsqueda en grilla, búsqueda aleatoria, Optuna, y un algoritmo genético (con una población de 6 individuos). 

## 5. Resultados y métricas clave

En la **base mundial**, Random Forest y XGBoost, junto con el SVM de kernel, logran los mejores resultados (F1 macro alrededor de 0.82 y R² sobre 0.80). Aplastan a los modelos lineales. Pero como vimos en el EDA, parte de este éxito viene de que estos modelos logran identificar la región de la planta. Si quitamos la latitud y longitud, el desempeño cae en picada. 

En la **base de Colombia**, la historia se invierte. El factor de capacidad (FC) es casi proporcional a la radiación que recibe el panel. Por eso, un modelo lineal como Lasso logra explicar un R² de 0.47 del sube y baja diario de la planta, igualando o superando a modelos más complejos que tienden a sobreajustarse a las 6 zonas geográficas disponibles. Vimos empíricamente que cada kWh/m² de radiación suma alrededor de 4.7 puntos al FC.

## 6. Las pruebas estadísticas

Para salir de dudas, aplicamos la prueba de Clark-West y mediciones como la d de Cohen y la prueba de Diebold-Mariano (1995). Los resultados confirmaron sin espacio a duda la superioridad de Random Forest en la base global, con una d de Cohen gigantesca (10.01 respecto a Ridge y Lasso). Clark-West probó de forma contundente que incluir latitud y longitud dispara el poder predictivo (p < 0.001).

## 7. Qué aprendimos sobre los modelos (Interpretabilidad y Rendimiento)

El perfilamiento computacional confirmó que KNN es rápido de entrenar pero muy lento en inferencia. Random Forest y XGBoost concentran su costo al entrenar, logrando evaluar en fracciones de segundo, con picos de memoria bajísimos (el cuello de botella es el procesador, no la RAM). 

Al aplicar técnicas de explicabilidad, notamos que SHAP y LIME ordenan de forma bastante diferente la cola de importancia de variables, pero ambos coinciden en cuáles son los 3 factores más críticos. 

## 8. Conclusiones y lo que queda pendiente

El índice del dataset mundial no mide aptitud desde el clima, sino principalmente la región. Los modelos pueden predecirlo muy bien si "saben" dónde están, pero fracasan en continentes nuevos. Por otro lado, en Colombia confirmamos que la radiación sí explica la mitad del comportamiento diario de una planta real, y un modelo lineal simple basta para capturarlo. 

Queda pendiente medir la tecnología específica de cada planta en Colombia (potencia DC real, tipo de seguidor), lo cual probablemente explicaría la gran diferencia de nivel base que hay entre una planta y otra. Además, trabajar con radiación medida por estaciones en terreno podría romper el techo del R² de 0.50 que nos impone la radiación estimada por satélite.


## 12. Referencias

Todas verificadas contra Crossref o DataCite el 2 de octubre de 2026. Los BibTeX están en `referencias.bib`. Las entradas sin DOI llevan la URL. Antes de usar una cita en el informe final hay que leer el texto completo.

- Enel Green Power. (2023). *El parque solar Fundación, construido por Enel Green Power en el departamento del Magdalena, inicia su etapa de pruebas*. https://www.enelgreenpower.com/es/medios/press/2023/09/egp-colombia-inicio-etapa-de-pruebas-parque-solar-fundacion
- Faiman, D. (2008). Assessing the outdoor operating temperature of photovoltaic modules. *Progress in Photovoltaics*. https://doi.org/10.1002/pip.813
- Field, C. A. y Welsh, A. H. (2007). Bootstrapping clustered data. *Journal of the Royal Statistical Society B*. https://doi.org/10.1111/j.1467-9868.2007.00593.x
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*. https://doi.org/10.1214/aos/1013203451
- Global Energy Monitor. (2026). *Global Solar Power Tracker, February 2026 release*. https://globalenergymonitor.org/projects/global-solar-power-tracker
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society, 146*(730), 1999-2049. https://doi.org/10.1002/qj.3803
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343 (preprint arXiv:2603.20601)
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Muñoz-Sabater, J. et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications. *Earth System Science Data, 13*(9), 4349-4383. https://doi.org/10.5194/essd-13-4349-2021
- pv magazine LATAM. (2023, 15 de marzo). *Avanza en Colombia el parque solar Fundación, de 132 MW*. https://www.pv-magazine-latam.com/2023/03/15/avanza-en-colombia-el-parque-solar-fundacion-de-132-mw/
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Skoplaki, E. y Palyvos, J. A. (2009). On the temperature dependence of photovoltaic module electrical performance: a review of efficiency/power correlations. *Solar Energy, 83*(5), 614-624. https://doi.org/10.1016/j.solener.2008.10.008
- Skoplaki, E. y Palyvos, J. A. (2009). Operating temperature of photovoltaic modules: a survey of pertinent correlations. *Renewable Energy, 34*(1), 23-29. https://doi.org/10.1016/j.renene.2008.04.009
- Urraca, R. et al. (2018). Evaluation of global horizontal irradiance estimates from ERA5 and COSMO-REA6 reanalyses using ground and satellite-based data. *Solar Energy, 164*, 339-354. https://doi.org/10.1016/j.solener.2018.02.059
- XM S.A. E.S.P. (2026). *API pública de datos del mercado de energía mayorista y portal SiMEM*. https://www.simem.co/
- Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [software]. Zenodo. https://doi.org/10.5281/zenodo.7970649